# AdharaAI — InLegalBERT Fine-Tuning Prep

This notebook prepares the full fine-tuning pipeline for **InLegalBERT** ahead of receiving real labeled clause data from P3.

**Goal:** predict two things per clause:
- `risk_level` — high / medium / low
- `clause_type` — the category of risk (e.g. `unilateral_termination`, `non_refundable_deposit`, etc. — matching your `risk_flagger.py` rule IDs where possible)

**Why prep now with synthetic data:** once P3 delivers `sample_labels.csv` in the real 7-column format (`clause_text, risk_level, clause_type, risk_reason, simplified_text, tip, source_file`), you just swap the CSV path in Section 2 and rerun — everything else stays the same.

**Runtime:** Make sure you're on a GPU runtime — `Runtime > Change runtime type > T4 GPU` (free tier).


In [1]:
# ── Section 0: Mount Google Drive (run this FIRST, before anything else) ─────
# This connects your Google Drive to this Colab session so that anything you
# save under /content/drive/MyDrive/... survives after you close the tab.
# Colab's local disk (everything outside /content/drive) is WIPED when the
# session ends — Drive is the only thing that persists.
#
# Running this cell will prompt a popup asking you to sign in and grant
# access — click through it once per session.

from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/AdharaAI_InLegalBERT"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Model and checkpoints will be saved to: {SAVE_DIR}")


Mounted at /content/drive
Model and checkpoints will be saved to: /content/drive/MyDrive/AdharaAI_InLegalBERT


In [ ]:
# ── Section 1: Install dependencies ──────────────────────────────────────────
!pip install -q transformers datasets scikit-learn accelerate


In [5]:
# ── Section 2: Load data (104-row deployment-ready dataset) ──────────────────
import pandas as pd
from google.colab import files

uploaded = files.upload()   # select adharaai_deployment_ready.csv (the 104-row version)

df = pd.read_csv('adharaai_deployment_ready_merged.csv')

print(f"Loaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"Risk levels: {df['risk_level'].value_counts().to_dict()}")
print(f"Clause types: {df['clause_type'].value_counts().to_dict()}")
print(df[['clause_text', 'risk_level', 'clause_type']].head())

Saving adharaai_deployment_ready_merged.csv to adharaai_deployment_ready_merged (1).csv
Loaded 104 rows
Columns: ['clause_text', 'risk_level', 'clause_type', 'risk_reason', 'simplified_text', 'tip', 'source_file']
Risk levels: {'low': 39, 'medium': 33, 'high': 32}
Clause types: {'termination': 19, 'payment': 19, 'amendment': 18, 'access': 17, 'penalty': 10, 'restriction': 9, 'dispute': 7, 'maintenance': 5}
                                         clause_text risk_level  clause_type
0  The rent in respect of the Demised Premises sh...        low  termination
1  The Tenant shall pay to the Owner a monthly re...        low      payment
2  The Tenant shall pay to the Owner a monthly ma...     medium  maintenance
3  The Tenant will pay to the Owner an interest-f...        low      payment
4  The said amount of the Security deposit shall ...     medium      payment


In [6]:
# ── Section 3: Encode labels ─────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

risk_encoder = LabelEncoder()
type_encoder = LabelEncoder()

df["risk_label_id"] = risk_encoder.fit_transform(df["risk_level"])
df["type_label_id"] = type_encoder.fit_transform(df["clause_type"])

num_risk_labels = len(risk_encoder.classes_)
num_type_labels = len(type_encoder.classes_)

print("Risk level classes:", list(risk_encoder.classes_))
print("Clause type classes:", list(type_encoder.classes_))
print(f"num_risk_labels={num_risk_labels}, num_type_labels={num_type_labels}")

# NOTE: with real data, num_type_labels will grow substantially (dozens of
# clause types matching your risk_flagger.py rule IDs). That's expected —
# just rerun this cell after loading the real CSV.


Risk level classes: ['high', 'low', 'medium']
Clause type classes: ['access', 'amendment', 'dispute', 'maintenance', 'payment', 'penalty', 'restriction', 'termination']
num_risk_labels=3, num_type_labels=8


In [7]:
# ── Section 4: Train/val split + tokenization ────────────────────────────────
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "law-ai/InLegalBERT"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df, val_df = train_test_split(
    df, test_size=0.25, random_state=42, stratify=df["risk_label_id"]
)

print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows")
# 104-row real dataset — this split is meaningful, not a sanity check.

def make_dataset(sub_df):
    ds = Dataset.from_pandas(sub_df[["clause_text", "risk_label_id", "type_label_id"]].reset_index(drop=True))
    def tokenize_fn(batch):
        return tokenizer(batch["clause_text"], truncation=True, padding="max_length", max_length=256)
    ds = ds.map(tokenize_fn, batched=True)
    return ds

train_ds = make_dataset(train_df)
val_ds = make_dataset(val_df)


config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Train: 78 rows | Val: 26 rows


Map:   0%|          | 0/78 [00:00<?, ? examples/s]

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

In [8]:
# ── Section 5: Two-head model (risk_level + clause_type) ─────────────────────
import torch
import torch.nn as nn
from transformers import AutoModel, AutoConfig

class InLegalBERTMultiHead(nn.Module):
    def __init__(self, model_name, num_risk_labels, num_type_labels):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.risk_head = nn.Linear(hidden, num_risk_labels)
        self.type_head = nn.Linear(hidden, num_type_labels)

    def forward(self, input_ids=None, attention_mask=None,
                risk_label_id=None, type_label_id=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]  # [CLS] token
        pooled = self.dropout(pooled)

        risk_logits = self.risk_head(pooled)
        type_logits = self.type_head(pooled)

        loss = None
        if risk_label_id is not None and type_label_id is not None:
            loss_fn = nn.CrossEntropyLoss()
            risk_loss = loss_fn(risk_logits, risk_label_id)
            type_loss = loss_fn(type_logits, type_label_id)
            loss = risk_loss + type_loss  # equal weighting; tune later if needed

        return {"loss": loss, "risk_logits": risk_logits, "type_logits": type_logits}


model = InLegalBERTMultiHead(MODEL_NAME, num_risk_labels, num_type_labels)
print(model)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  534MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  534MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


InLegalBERTMultiHead(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementw

In [11]:
from transformers import TrainingArguments, Trainer
import numpy as np

class MultiHeadCollator:
    def __call__(self, features):
        batch = {
            "input_ids":      torch.tensor([f["input_ids"] for f in features]),
            "attention_mask": torch.tensor([f["attention_mask"] for f in features]),
            "risk_label_id":  torch.tensor([f["risk_label_id"] for f in features]),
            "type_label_id":  torch.tensor([f["type_label_id"] for f in features]),
        }
        return batch

def compute_metrics(eval_pred):
    (risk_logits, type_logits), labels = eval_pred
    risk_labels = labels[0] if isinstance(labels, (list, tuple)) else labels
    type_labels = labels[1] if isinstance(labels, (list, tuple)) else labels

    risk_preds = np.argmax(risk_logits, axis=1)
    type_preds = np.argmax(type_logits, axis=1)

    from sklearn.metrics import f1_score
    risk_f1 = f1_score(risk_labels, risk_preds, average="weighted", zero_division=0)
    type_f1 = f1_score(type_labels, type_preds, average="weighted", zero_division=0)
    return {"risk_f1": risk_f1, "type_f1": type_f1}

training_args = TrainingArguments(
    output_dir=f"{SAVE_DIR}/checkpoints",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=10,
    weight_decay=0.01,
    eval_strategy="epoch",        # ← fixed: eval_strategy not evaluation_strategy
    save_strategy="epoch",
    save_total_limit=2,          # keep only best 2 checkpoints (prevents disk crashes)
    load_best_model_at_end=True,
    metric_for_best_model="risk_f1",
)

model = InLegalBERTMultiHead("law-ai/InLegalBERT", num_risk_labels, num_type_labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=MultiHeadCollator(),
    compute_metrics=compute_metrics,
)

print("Trainer ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainer ready


In [12]:
# ── Section 7: Train on real 104-row dataset ──────────────────────────────

trainer.train()


Epoch,Training Loss,Validation Loss,Risk F1,Type F1
1,No log,3.149138,0.213675,0.010989
2,No log,2.830945,0.615100,0.336131
3,No log,2.507089,0.500000,0.561538
4,No log,2.281364,0.474725,0.664366
5,No log,2.184947,0.431463,0.649573


TrainOutput(global_step=100, training_loss=2.3330838012695314, metrics={'train_runtime': 113.7296, 'train_samples_per_second': 3.429, 'train_steps_per_second': 0.879, 'total_flos': 0.0, 'train_loss': 2.3330838012695314, 'epoch': 5.0})

In [13]:
# ── Section 8: Evaluate ────────────────────────────────────────────────────────
metrics = trainer.evaluate()
print(metrics)


Training Loss,Validation Loss,Epoch,Risk F1,Type F1
No log,2.830945,5,0.615100,0.336131


{'eval_loss': 2.8309454917907715, 'eval_risk_f1': 0.6150998322944025, 'eval_type_f1': 0.3361305361305362}


In [15]:
# ── Section 9: Save model + label encoders (to Google Drive, so it persists) ─
import pickle

FINAL_DIR = f"{SAVE_DIR}/final_model"
os.makedirs(FINAL_DIR, exist_ok=True)

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

with open(f"{FINAL_DIR}/label_encoders.pkl", "wb") as f:
    pickle.dump({"risk_encoder": risk_encoder, "type_encoder": type_encoder}, f)

print(f"Model, tokenizer, and label encoders saved to: {FINAL_DIR}")
print("This is inside Google Drive, so it will still be there next time you open Colab.")
print("To bring it into your local AdharaAI project: open the folder in Google Drive,")
print("download it as a zip, then unzip into your project (e.g. backend/models/inlegalbert/).")


Model, tokenizer, and label encoders saved to: /content/drive/MyDrive/AdharaAI_InLegalBERT/final_model
This is inside Google Drive, so it will still be there next time you open Colab.
To bring it into your local AdharaAI project: open the folder in Google Drive,
download it as a zip, then unzip into your project (e.g. backend/models/inlegalbert/).


## Next steps once P3's real data arrives

1. Replace the synthetic `df` in **Section 2** with `pd.read_csv('sample_labels.csv')`
2. Rerun Sections 3–9 top to bottom — no other code changes needed
3. Check `type_f1` and `risk_f1` from Section 8 — if `clause_type` classes are too imbalanced (some types with very few examples), consider grouping rare types or gathering more labeled examples for those specific categories
4. Once you're happy with metrics, the model in `./adharaai_inlegalbert_final` is ready to integrate as a second signal alongside your rule engine — e.g. flag a clause if *either* the rules or the model flags it as medium/high risk, and use the model's `clause_type` prediction as a fallback when no rule matches
